# Torch

In [1]:
import torch

print(torch.__version__)

2.11.0


In [2]:
print(torch.cuda.is_available())

False


## scalar, vector, matrices and tensor

In [3]:
import numpy as np

# 0D scalar
tensor0d = torch.tensor(2)
tensor0d

tensor(2)

In [4]:
# 1D vector
tensor1D = torch.tensor([1,2,3])
tensor1D

tensor([1, 2, 3])

In [5]:
# 2D matrices
tensor2D = torch.tensor([[1,2]
                         ,[2,3]])
tensor2D

tensor([[1, 2],
        [2, 3]])

In [6]:
# 3D tensor
tensor3D = torch.tensor([[[1,1,1],
                          [2,2,2]],
                         [[3,3,3],
                          [4,4,4]]])

tensor3D

tensor([[[1, 1, 1],
         [2, 2, 2]],

        [[3, 3, 3],
         [4, 4, 4]]])

### 2 ways to use torch with numpy array value
1. copies numpy array
2. shares memory with numpy array

In [12]:
# let say we have a 1x2x2 (3D) numpy array
array3D = np.array([[[1,1,1],
                     [2,2,2]],
                    [[3,3,3],
                     [4,4,4]]])
array3D

array([[[1, 1, 1],
        [2, 2, 2]],

       [[3, 3, 3],
        [4, 4, 4]]])

In [13]:
# using first approach. tensor just copy that array
array_tensor1 = torch.tensor(array3D)
array_tensor1

tensor([[[1, 1, 1],
         [2, 2, 2]],

        [[3, 3, 3],
         [4, 4, 4]]])

In [15]:
array3D[0, 0, 0] = 999
array_tensor1 # remains unchanged

tensor([[[1, 1, 1],
         [2, 2, 2]],

        [[3, 3, 3],
         [4, 4, 4]]])

In [16]:
# however if using second approach
array_tensor2 = torch.from_numpy(array3D)
array_tensor2 # immediately changed

tensor([[[999,   1,   1],
         [  2,   2,   2]],

        [[  3,   3,   3],
         [  4,   4,   4]]])

### Tensor data types

In [19]:
# another tips to see types on tensor
tensor1d = torch.tensor([1.1,2.2,3.3])
tensor1d.dtype

torch.float32

In [21]:
# orr
tensor1d = torch.tensor([1,2,3])
tensor1d.to(torch.float32)

tensor([1., 2., 3.])

### basic pytorch operation
1. create
2. shape
3. reshape
4. view
5. T, transpose
6. matmul, multiplication

In [22]:
# create tensor. say 2x3 2D
tensor2D = torch.tensor([[1,2,3]
                         ,[4,5,6]])

In [23]:
# now let see the same
tensor2D.shape

torch.Size([2, 3])

In [26]:
# see? 2x3 2D tensor, easy.. now lets reshape it
tensor2D.reshape(3,2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [28]:
# its still 2D, but its transform from 2x3 to 3x2
# new let say, we want to just see the transformation without changin the real shape
tensor2D.view(2,3)

tensor([[1, 2, 3],
        [4, 5, 6]])

In [29]:
# soo fricking ez. now lets do some transpose, with just .T, wtf
tensor2D.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [30]:
# transpose basically change column value to row value and vice versa. (i,j) -> (j,i)
# of course the most important thing we are gonna using is just this...
tensor2D.matmul(tensor2D.T)

tensor([[14, 32],
        [32, 77]])

In [31]:
# multiply matrices rule is basically just make sure column on 1st tensor is the same as row on 2nd array. thats it
# matmul or @ its basically the same
tensor2D @ tensor2D.T

tensor([[14, 32],
        [32, 77]])

### Usage on Neural network

sayy, we are doing single forward propagation.
soo, you know the basic formula for this

- loss = -[ y log(σ(w·x + b)) + (1 - y) log(1 - σ(w·x + b)) ]

its basically come from this:
- x => w·x + b => sigmoid => a => compare with y => loss

Sooo. here is our plan

1. Input & parameter
- x = 1
- w = 2
- b = -1
- y = 0

2. Linear (z = w·x + b)

- z = (2 × 1) + (-1) = 1

3. Sigmoid (a = σ(z))

- σ(z) = 1 / (1 + e^{-z})
- Input z = 1:
- a = 1 / (1 + e^{-1}) ≈ 0.73

4. Loss (Binary Cross Entropy)
- cause y = 1, rumusnya jadi:
- loss = - log(1 - a)
- input a ≈ 0.73:
- loss = -log(1 - 0.73)
     = -log(0.27)
     ≈ 1.31

soo. 
- z = 1
- a ≈ 0.73
- loss ≈ 1.31


In [42]:
# lets implement using pytorch, note: requires_grad means it can be derivied
from torch.autograd import grad
import torch.nn.functional as F

y = torch.tensor([1]).to(torch.float32)
x = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True) 
b = torch.tensor([0.0], requires_grad=True)

In [69]:
# linear, x . w1 + b
z = x * w1 + b 
z

tensor([2.4200], grad_fn=<AddBackward0>)

In [70]:
# activation function. for what?? for make it more stable or normalize it
a = torch.sigmoid(z)
a

tensor([0.9183], grad_fn=<SigmoidBackward0>)

In [71]:
# now lets calculate the loss, what is loss? bruh, its just comparing to real value (approximation : actual)
loss = F.binary_cross_entropy(a,y)
loss

tensor(0.0852, grad_fn=<BinaryCrossEntropyBackward0>)

In [72]:
# okey lets add some initial backprops, i knoww. but lets just test it.
# so here is the drill. just finding derivative of loss value to w or b 
# is just to know the direciton of how our w or b need to be chaged. grad + -> decrease weight, grad - -> increase weight
# the formula?? bruf, here.. -> dL/dw = (dL/da) × (da/dz) × (dz/dw) or (dz/db) for b
grad_l_w1 = grad(loss, w1, retain_graph=True)

grad_l_w1

(tensor([-0.0898]),)

In [73]:
# now lets see for the b
grad_l_b = grad(loss, b, retain_graph=True)
grad_l_b

(tensor([-0.0817]),)

In [74]:
# orrr, for efficiency or you already understand the process, just use this. same process
loss.backward()

grad_l_w1, grad_l_b

((tensor([-0.0898]),), (tensor([-0.0817]),))

In [75]:
# see?? samee.. oh yaa
# later, if you curious. this value multiplied by lr and being an updated b or w value
# andd, process repeated again and again untill a = y orr, close..

### Usage on more complex NN operation 
**(multilayer/multi b/multi w/ multi operation/multi L)**

In [76]:
# so here is the deal, we are gonna using 50, yeapp 50 fatures input. to just predict 3 output. 
# 3 layer , output counts. 
# first layer will have input 1x50 @ 50x30 weight + 1x30 b -> 1x30
# second layer, 1x30 @ 30x20 w + 1x20 b -> 1x20
# third layer (output), 1x20 @ 20x3 -> 1x3 = ([logits1, logits2, logits3])

In [80]:
# lets implement that
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            
            # layer 1
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # layer 2
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # layer 3
            torch.nn.Linear(20, num_outputs),
            torch.nn.ReLU(),
            
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [81]:
multilayer_nn_model = NeuralNetwork(50,3)
multilayer_nn_model

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
    (5): ReLU()
  )
)

In [83]:
# now lets see how much params we use here
num_params = sum(p.numel() for p in multilayer_nn_model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [84]:
# dont worry its basically 50x30 + 30 = 1530 from layer 1
# 30x20 + 20 = 620 from layer 2
# andd 20x3 + 3 = 63 from layer output. sooo, add all of them. we get 2213

In [86]:
# now let see first layer weight
multilayer_nn_model.layers[0].weight

Parameter containing:
tensor([[-0.0805,  0.1393,  0.1113,  ...,  0.0870, -0.1104,  0.1156],
        [ 0.0893,  0.0983, -0.0996,  ..., -0.1067, -0.0227,  0.0359],
        [-0.0049, -0.1243, -0.0794,  ...,  0.0763,  0.0050, -0.0635],
        ...,
        [-0.0882,  0.1385,  0.0945,  ...,  0.0872, -0.0018, -0.0908],
        [-0.1014, -0.1032, -0.0493,  ...,  0.0404, -0.0777,  0.0152],
        [-0.0462,  0.0337, -0.1245,  ...,  0.0039, -0.0092,  0.0121]],
       requires_grad=True)

In [87]:
multilayer_nn_model.layers[0].weight.shape

torch.Size([30, 50])

In [89]:
# see?? 30x50, wait a minute. oh yeah i forgot. for specific layer or nn operation. pytorch swapping it like that
# but the actual size is 50x30. dont worry. for b, same thing, oh noo, bias will same like usual. 
multilayer_nn_model.layers[0].bias.shape

torch.Size([30])

In [91]:
# we can also init tha value to not always random everytime we run that function. using?? yess seed
torch.manual_seed(42)

model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[ 0.1081,  0.1174, -0.0331,  ...,  0.0253,  0.0718, -0.0862],
        [-0.1400, -0.0546, -0.1085,  ..., -0.0477, -0.0501, -0.1368],
        [-0.0810,  0.0353, -0.0187,  ...,  0.1142,  0.1288, -0.1121],
        ...,
        [-0.0031, -0.0573,  0.0515,  ...,  0.0271, -0.0928, -0.1175],
        [-0.0444, -0.1318, -0.0660,  ...,  0.0647, -0.1230, -0.0531],
        [ 0.0023, -0.1223,  0.0797,  ...,  0.0369,  0.0862,  0.1328]],
       requires_grad=True)


In [92]:
# dont worry about params number on that seed function. every number is basically the same purpose.

In [93]:
# okey now we have our model with initial random weight and bias (parameters). lets input some input
# remember, input will have 1x50 shape. and we just pass it into that model variable it self. 
torch.manual_seed(42)
X = torch.rand((1, 50))
approx = model(X)
approx

tensor([[0.1215, 0.0000, 0.0000]], grad_fn=<ReluBackward0>)

In [94]:
# sure, we have logits now. but its still means nothing. we need to map it to probability of each class
# we have 3 classes. our goal is based on that 50 feature, params can predict just one classes.
# take example, 50 feature of cats. 4 legs etc. after it optimal, which is possibility is 1 for cat and 0,0 for another.
# that params can be reusable for future inputs to predict whats cat or not
with torch.no_grad():
    possibility = torch.softmax(approx, dim=1)
possibility

tensor([[0.3608, 0.3196, 0.3196]])

In [96]:
# that with and bla bla is just efficiency computation
# but its basically the same as 
possibility = torch.softmax(approx, dim=1)
possibility

tensor([[0.3608, 0.3196, 0.3196]], grad_fn=<SoftmaxBackward0>)

In [97]:
# if we can see. with that 50 feature of cats. model still cant predict its cat.
# its have same proportion,which mean still not optimal. but thats okey cause its just forward props.
# okey, we have our forward props function. how tf we doing the backprops or derivative it and update params
# till output is confident?? which is [1,0,0]